In [1]:

# Import required libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

# Load the customer churn dataset
df = pd.read_csv("/content/customer-churn-training.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())


Dataset loaded successfully!
Dataset shape: (12, 7)
Columns: ['customer_id', 'tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days', 'plan_type', 'churned']


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0


In [3]:

# Separate input features and target
X = df.drop(columns=["customer_id", "churned"])
y = df["churned"]

# Identify numerical and categorical columns
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical features:", numeric_cols)
print("Categorical features:", categorical_cols)

print("\nTarget distribution:")
print(y.value_counts())


Numerical features: ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days']
Categorical features: ['plan_type']

Target distribution:
churned
0    7
1    5
Name: count, dtype: int64


In [4]:

# Split data before preprocessing to avoid data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


Training rows: 9
Testing rows: 3


In [5]:

# Preprocess numerical and categorical features separately

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

print("Preprocessing pipeline created successfully!")


Preprocessing pipeline created successfully!


In [6]:

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        random_state=42
    ),
    "Extra Trees": ExtraTreesClassifier(
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

print("Four classification models defined successfully!")


Four classification models defined successfully!


In [7]:

results = []

for name, classifier in models.items():
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, predictions, zero_division=0
        )
    })

results_df = pd.DataFrame(results)
display(results_df.sort_values("F1 Score", ascending=False))

print("Model comparison completed!")


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,1.0,1.0,1.0,1.0
1,Random Forest,1.0,1.0,1.0,1.0
2,Extra Trees,1.0,1.0,1.0,1.0
3,Gradient Boosting,1.0,1.0,1.0,1.0


Model comparison completed!


In [8]:

# Use stratified cross-validation on the training data
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

param_grid = {
    "classifier__n_estimators": [50, 100],
    "classifier__max_depth": [None, 3, 5],
    "classifier__min_samples_split": [2, 4]
}

grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation F1 score:", grid_search.best_score_)


Best parameters: {'classifier__max_depth': None, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 50}
Best cross-validation F1 score: 1.0


In [9]:

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print("Tuned Random Forest Evaluation")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(
    y_test, y_pred, zero_division=0
))
print("Recall:", recall_score(
    y_test, y_pred, zero_division=0
))
print("F1 Score:", f1_score(
    y_test, y_pred, zero_division=0
))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred, zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Tuned Random Forest Evaluation
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3

Confusion Matrix:
[[2 0]
 [0 1]]


In [10]:

feature_names = best_model.named_steps[
    "preprocessor"
].get_feature_names_out()

importances = best_model.named_steps[
    "classifier"
].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

display(importance_df)

print("Feature importance analysis completed!")


,Feature,Importance
1,numeric__support_tickets,0.299500
0,numeric__tenure_months,0.287500
4,categorical__plan_type_Basic,0.147500
2,numeric__monthly_spend_inr,0.140000
3,numeric__last_login_days,0.072857
5,categorical__plan_type_Pro,0.040143
6,categorical__plan_type_Standard,0.012500


Feature importance analysis completed!


In [11]:

best_comparison = results_df.sort_values(
    "F1 Score", ascending=False
).iloc[0]

print("Best model based on test F1 score:")
print("Model:", best_comparison["Model"])
print("F1 Score:", best_comparison["F1 Score"])

print("\nTuned Random Forest cross-validation F1 score:")
print(grid_search.best_score_)


Best model based on test F1 score:
Model: Logistic Regression
F1 Score: 1.0

Tuned Random Forest cross-validation F1 score:
1.0


In [12]:

import joblib

joblib.dump(best_model, "best_customer_churn_model.pkl")

print("Final tuned model saved successfully!")


Final tuned model saved successfully!


In [13]:

from google.colab import files

files.download("best_customer_churn_model.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>